# 07 — Detector datasets (COCO) + splits + QC

Document your COCO tooling and dataset locations for training detectors.


In [17]:
from __future__ import annotations

import os, sys, subprocess, json, re
from pathlib import Path
from datetime import datetime

REPO_ROOT = Path.cwd().parent
print("REPO_ROOT:", REPO_ROOT)

def sh(cmd: str, check: bool=True) -> None:
    """Run a shell command (prints it first) from REPO_ROOT."""
    print("\n▶", cmd)
    subprocess.run(cmd, shell=True, check=check, cwd=REPO_ROOT)

def pick_first_existing(*cands: str) -> Path:
    for c in cands:
        p = Path(c)
        if p.exists():
            return p
    return Path(cands[0])

def require_exists(p: Path, what: str="path") -> Path:
    if not p.exists():
        raise FileNotFoundError(f"Missing {what}: {p}")
    return p

def newest_path(glob_pat: str) -> Path | None:
    paths = list(REPO_ROOT.glob(glob_pat))
    if not paths:
        return None
    paths.sort(key=lambda p: p.stat().st_mtime, reverse=True)
    return paths[0]

def show_tree(root: Path, max_lines: int=200) -> None:
    i = 0
    for p in sorted(root.rglob("*")):
        if i >= max_lines:
            print("... (truncated)")
            return
        if p.is_dir():
            continue
        rel = p.relative_to(root)
        print(rel)
        i += 1

for name, subdir in {
    "sanitize_coco.py": "src/tools",
    "split_coco.py": "src/torchvision_det",              # adjust if needed
    "rebuild_coco_from_split.py": "src/torchvision_det",
}.items():
    p = REPO_ROOT / subdir / name
    print(str(p), "->", "OK" if p.exists() else "MISSING")


REPO_ROOT: /Users/ameerfiras/REDNET-ML
/Users/ameerfiras/REDNET-ML/src/tools/sanitize_coco.py -> OK
/Users/ameerfiras/REDNET-ML/src/torchvision_det/split_coco.py -> OK
/Users/ameerfiras/REDNET-ML/src/torchvision_det/rebuild_coco_from_split.py -> OK


## 7.1 Print help for COCO tools


In [18]:
# --- sanity check scripts exist ---
for name, subdir in {
    "sanitize_coco.py": "src/tools",
    "split_coco.py": "src/torchvision_det",
    "rebuild_coco_from_split.py": "src/torchvision_det",
}.items():
    p = REPO_ROOT / subdir / name
    print(p, "->", "OK" if p.exists() else "MISSING")
    if p.exists():
        sh(f'python "{p}" --help', check=False)

# --- run split_coco with your real dataset ---
IMG_ROOT = REPO_ROOT / "training/labels/detection/images"   
LBL_ROOT = REPO_ROOT / "training/labels"                   
OUT_DIR  = REPO_ROOT / "training/splits"

OUT_DIR.mkdir(parents=True, exist_ok=True)

print("\nRunning COCO split...")
sh(
    f'python "{REPO_ROOT / "src/torchvision_det/split_coco.py"}" '
    f'--img-root "{IMG_ROOT}" '
    f'--lbl-root "{LBL_ROOT}" '
    f'--out "{OUT_DIR}"'
)

print("✓ COCO split complete")


/Users/ameerfiras/REDNET-ML/src/tools/sanitize_coco.py -> OK

▶ python "/Users/ameerfiras/REDNET-ML/src/tools/sanitize_coco.py" --help
/Users/ameerfiras/REDNET-ML/src/torchvision_det/split_coco.py -> OK

▶ python "/Users/ameerfiras/REDNET-ML/src/torchvision_det/split_coco.py" --help
train: 267 images, 400 anns
val: 33 images, 51 anns
test: 34 images, 47 anns
/Users/ameerfiras/REDNET-ML/src/torchvision_det/rebuild_coco_from_split.py -> OK

▶ python "/Users/ameerfiras/REDNET-ML/src/torchvision_det/rebuild_coco_from_split.py" --help
usage: rebuild_coco_from_split.py [-h] [--img-root IMG_ROOT]
                                  [--lbl-root LBL_ROOT] [--out OUT]

options:
  -h, --help           show this help message and exit
  --img-root IMG_ROOT
  --lbl-root LBL_ROOT
  --out OUT

Running COCO split...

▶ python "/Users/ameerfiras/REDNET-ML/src/torchvision_det/split_coco.py" --img-root "/Users/ameerfiras/REDNET-ML/training/labels/detection/images" --lbl-root "/Users/ameerfiras/REDNET-ML/tra

## 7.2 List candidate dataset directories


In [20]:

for root in ["training", "runs/detect", "src/torchvision_det/training"]:
    p = REPO_ROOT / root
    if p.exists():
        print("\n", root)
        for sub in sorted(p.glob("*"))[:60]:
            print(" -", sub)



 training
 - /Users/ameerfiras/REDNET-ML/training/aerial_summer_2017
 - /Users/ameerfiras/REDNET-ML/training/aerial_summer_2018
 - /Users/ameerfiras/REDNET-ML/training/aerial_summer_2019
 - /Users/ameerfiras/REDNET-ML/training/aerial_summer_2020
 - /Users/ameerfiras/REDNET-ML/training/aerial_summer_2021
 - /Users/ameerfiras/REDNET-ML/training/aerial_summer_2022
 - /Users/ameerfiras/REDNET-ML/training/aerial_summer_2023
 - /Users/ameerfiras/REDNET-ML/training/aerial_summer_2024
 - /Users/ameerfiras/REDNET-ML/training/aerial_winter_2017
 - /Users/ameerfiras/REDNET-ML/training/aerial_winter_2018
 - /Users/ameerfiras/REDNET-ML/training/aerial_winter_2019
 - /Users/ameerfiras/REDNET-ML/training/aerial_winter_2020
 - /Users/ameerfiras/REDNET-ML/training/aerial_winter_2021
 - /Users/ameerfiras/REDNET-ML/training/aerial_winter_2022
 - /Users/ameerfiras/REDNET-ML/training/aerial_winter_2023
 - /Users/ameerfiras/REDNET-ML/training/aerial_winter_2024
 - /Users/ameerfiras/REDNET-ML/training/detec